# 面试题：工程里怎样手写并正确使用最短路算法？

## 可以直接复述的回答

图算法落地的第一步是把实体、边方向、权重含义和不可用状态定义清楚，而不是先选库。BFS 只保证最少边数，在每条边成本不同的配送网络中不保证最短时间；非负权图应使用 Dijkstra。正确的 Dijkstra 在节点以当前最小距离从堆中弹出时才最终确认距离，并跳过陈旧堆项，不能在第一次入堆时就标记 visited。工程输出应包含完整路径、总成本和搜索账本，方便解释路线为何改变。算法还要明确处理不可达、闭路、负权和动态权重。本题用 8 个物流节点、13 条带分钟权重的道路和 6 个配送请求，对比最少跳数基线与手写 Dijkstra，并复现“入堆即访问”导致的错误结果。

## 真实案例

样本模拟同城配送网络，边权表示脱敏后的预计行驶分钟，节点包含总仓、枢纽、站点和客户区域。它是静态小图教学实验，不包含真实地址，不能外推线上 ETA。

In [1]:
import heapq  # 导入最小堆以手写 Dijkstra 优先队列
from collections import defaultdict, deque  # 导入邻接表和 BFS 队列容器
from pprint import pprint  # 导入结构化打印函数以展示路径和搜索账本
edges = [("总仓", "北枢纽", 4), ("总仓", "南枢纽", 2), ("南枢纽", "北枢纽", 1), ("北枢纽", "东站", 5), ("南枢纽", "西站", 4), ("西站", "东站", 1), ("东站", "学校", 3), ("西站", "商场", 6), ("东站", "商场", 2), ("南枢纽", "医院", 7), ("西站", "医院", 2), ("北枢纽", "医院", 10), ("学校", "商场", 4)]  # 构造十三条双向配送道路及预计分钟
requests = [{"id": "R1", "起点": "总仓", "终点": "医院", "最优分钟": 8}, {"id": "R2", "起点": "总仓", "终点": "商场", "最优分钟": 9}, {"id": "R3", "起点": "北枢纽", "终点": "商场", "最优分钟": 7}, {"id": "R4", "起点": "南枢纽", "终点": "学校", "最优分钟": 8}, {"id": "R5", "起点": "西站", "终点": "学校", "最优分钟": 4}, {"id": "R6", "起点": "学校", "终点": "医院", "最优分钟": 6}]  # 定义六个带人工核验最优成本的配送请求
print("道路输入预览：")  # 输出真实图数据标题
pprint([{"from": left, "to": right, "分钟": minutes} for left, right, minutes in edges])  # 展示全部道路及权重
print("配送请求：")  # 输出评估样本标题
pprint(requests)  # 展示六组起终点和人工期望成本

道路输入预览：
[{'from': '总仓', 'to': '北枢纽', '分钟': 4},
 {'from': '总仓', 'to': '南枢纽', '分钟': 2},
 {'from': '南枢纽', 'to': '北枢纽', '分钟': 1},
 {'from': '北枢纽', 'to': '东站', '分钟': 5},
 {'from': '南枢纽', 'to': '西站', '分钟': 4},
 {'from': '西站', 'to': '东站', '分钟': 1},
 {'from': '东站', 'to': '学校', '分钟': 3},
 {'from': '西站', 'to': '商场', '分钟': 6},
 {'from': '东站', 'to': '商场', '分钟': 2},
 {'from': '南枢纽', 'to': '医院', '分钟': 7},
 {'from': '西站', 'to': '医院', '分钟': 2},
 {'from': '北枢纽', 'to': '医院', '分钟': 10},
 {'from': '学校', 'to': '商场', '分钟': 4}]
配送请求：
[{'id': 'R1', '最优分钟': 8, '终点': '医院', '起点': '总仓'},
 {'id': 'R2', '最优分钟': 9, '终点': '商场', '起点': '总仓'},
 {'id': 'R3', '最优分钟': 7, '终点': '商场', '起点': '北枢纽'},
 {'id': 'R4', '最优分钟': 8, '终点': '学校', '起点': '南枢纽'},
 {'id': 'R5', '最优分钟': 4, '终点': '学校', '起点': '西站'},
 {'id': 'R6', '最优分钟': 6, '终点': '医院', '起点': '学校'}]


## Baseline / 基线：BFS 找最少跳数

BFS 把每条道路都看成成本 1，因此返回经过边数最少的路线。下面仍计算该路线的真实分钟，便于和 Dijkstra 在同一指标上比较。

In [2]:
adjacency = defaultdict(list)  # 创建无向加权邻接表
edge_cost = {}  # 创建路径边到分钟成本的快速索引
for left, right, minutes in edges:  # 遍历全部道路记录
    adjacency[left].append((right, minutes))  # 加入正向邻居和权重
    adjacency[right].append((left, minutes))  # 加入反向邻居和权重
    edge_cost[(left, right)] = minutes  # 保存正向边成本
    edge_cost[(right, left)] = minutes  # 保存反向边成本
for node in adjacency:  # 遍历每个有道路的节点
    adjacency[node] = sorted(adjacency[node])  # 固定邻居顺序保证结果可复现
def route_minutes(path):  # 定义完整路径的真实分钟计算
    return sum(edge_cost[(path[index], path[index + 1])] for index in range(len(path) - 1)) if path else float("inf")  # 累加相邻道路权重
def bfs_fewest_hops(start, goal):  # 定义忽略权重的最少跳数基线
    queue = deque([(start, [start])])  # 初始化待访问节点和路径
    visited = {start}  # 记录已经进入队列的节点
    while queue:  # 按层处理所有可达节点
        current, path = queue.popleft()  # 取出最早入队的节点及路径
        if current == goal:  # 到达目标时立即返回最少跳数路径
            return path  # 返回基线路径
        for neighbor, _ in adjacency[current]:  # 遍历当前节点全部邻居但忽略权重
            if neighbor not in visited:  # 只把未访问节点加入队列
                visited.add(neighbor)  # 入队前标记避免重复
                queue.append((neighbor, path + [neighbor]))  # 保存邻居及扩展后的路径
    return []  # 不可达时返回空路径
baseline_rows = []  # 创建逐请求 BFS 结果表
for request in requests:  # 遍历六个配送请求
    path = bfs_fewest_hops(request["起点"], request["终点"])  # 计算最少道路数的路线
    baseline_rows.append({"请求": request["id"], "路径": path, "跳数": len(path) - 1, "真实分钟": route_minutes(path), "最优分钟": request["最优分钟"]})  # 保存路径及同口径成本
print("BFS 最少跳数 Baseline：")  # 输出基线标题
pprint(baseline_rows)  # 展示逐请求路径和时间损失

BFS 最少跳数 Baseline：
[{'最优分钟': 8, '真实分钟': 14, '请求': 'R1', '路径': ['总仓', '北枢纽', '医院'], '跳数': 2},
 {'最优分钟': 9, '真实分钟': 11, '请求': 'R2', '路径': ['总仓', '北枢纽', '东站', '商场'], '跳数': 3},
 {'最优分钟': 7, '真实分钟': 7, '请求': 'R3', '路径': ['北枢纽', '东站', '商场'], '跳数': 2},
 {'最优分钟': 8, '真实分钟': 9, '请求': 'R4', '路径': ['南枢纽', '北枢纽', '东站', '学校'], '跳数': 3},
 {'最优分钟': 4, '真实分钟': 4, '请求': 'R5', '路径': ['西站', '东站', '学校'], '跳数': 2},
 {'最优分钟': 6, '真实分钟': 18, '请求': 'R6', '路径': ['学校', '东站', '北枢纽', '医院'], '跳数': 3}]


## 手写核心：最小堆 Dijkstra 与松弛账本

堆项保存当前距离和节点；若弹出的距离不是最新 best distance，就跳过。每次发现更短路径时更新 distance、predecessor 并重新入堆，这一步称为松弛。

In [3]:
def dijkstra(start, goal):  # 定义适用于非负道路权重的最短路算法
    distances = {start: 0.0}  # 初始化起点距离为零
    predecessors = {}  # 保存最优路径上的前驱节点
    heap = [(0.0, start)]  # 创建以累计分钟排序的最小堆
    ledger = []  # 保存弹出节点和松弛动作供教学观察
    while heap:  # 持续处理仍可能改进的候选节点
        current_distance, current = heapq.heappop(heap)  # 弹出当前已知距离最小的节点
        if current_distance != distances.get(current):  # 检查该堆项是否已经被更短路线淘汰
            ledger.append({"动作": "跳过陈旧项", "节点": current, "堆中距离": current_distance, "最新距离": distances.get(current)})  # 记录陈旧堆项
            continue  # 不再从过时距离继续扩展
        ledger.append({"动作": "确认弹出", "节点": current, "距离": current_distance})  # 记录按最短距离确认的节点
        if current == goal:  # 目标首次以最新最小距离弹出时即可停止
            break  # 提前结束搜索
        for neighbor, weight in adjacency[current]:  # 遍历当前节点所有带权道路
            candidate_distance = current_distance + weight  # 计算经过当前节点到邻居的新距离
            if candidate_distance < distances.get(neighbor, float("inf")):  # 判断新路线是否真正更短
                old_distance = distances.get(neighbor, float("inf"))  # 保存被替换的旧距离
                distances[neighbor] = candidate_distance  # 更新邻居的最优累计分钟
                predecessors[neighbor] = current  # 记录最优路线的前驱
                heapq.heappush(heap, (candidate_distance, neighbor))  # 把改进后的候选重新压入堆
                ledger.append({"动作": "松弛", "边": (current, neighbor), "旧距离": old_distance, "新距离": candidate_distance})  # 记录关键中间量
    if goal not in distances:  # 检查目标是否不可达
        return [], float("inf"), ledger  # 返回空路径、无穷成本和访问账本
    path = [goal]  # 从目标开始反向重建最短路径
    while path[-1] != start:  # 持续回溯直到起点
        path.append(predecessors[path[-1]])  # 追加当前节点的最优前驱
    path.reverse()  # 把逆序路径翻转为起点到终点
    return path, distances[goal], ledger  # 返回最短路径、分钟和完整搜索账本
example_path, example_minutes, example_ledger = dijkstra("总仓", "医院")  # 运行会多次改进候选的示例请求
print("R1 的 Dijkstra 弹出与松弛账本：")  # 输出算法中间过程标题
pprint(example_ledger)  # 展示堆顺序、距离更新和陈旧项处理
print("R1 最短路径：", example_path, "分钟：", example_minutes)  # 展示从前驱表重建的最终路径

R1 的 Dijkstra 弹出与松弛账本：
[{'动作': '确认弹出', '节点': '总仓', '距离': 0.0},
 {'动作': '松弛', '新距离': 4.0, '旧距离': inf, '边': ('总仓', '北枢纽')},
 {'动作': '松弛', '新距离': 2.0, '旧距离': inf, '边': ('总仓', '南枢纽')},
 {'动作': '确认弹出', '节点': '南枢纽', '距离': 2.0},
 {'动作': '松弛', '新距离': 3.0, '旧距离': 4.0, '边': ('南枢纽', '北枢纽')},
 {'动作': '松弛', '新距离': 9.0, '旧距离': inf, '边': ('南枢纽', '医院')},
 {'动作': '松弛', '新距离': 6.0, '旧距离': inf, '边': ('南枢纽', '西站')},
 {'动作': '确认弹出', '节点': '北枢纽', '距离': 3.0},
 {'动作': '松弛', '新距离': 8.0, '旧距离': inf, '边': ('北枢纽', '东站')},
 {'动作': '跳过陈旧项', '堆中距离': 4.0, '最新距离': 3.0, '节点': '北枢纽'},
 {'动作': '确认弹出', '节点': '西站', '距离': 6.0},
 {'动作': '松弛', '新距离': 7.0, '旧距离': 8.0, '边': ('西站', '东站')},
 {'动作': '松弛', '新距离': 8.0, '旧距离': 9.0, '边': ('西站', '医院')},
 {'动作': '松弛', '新距离': 12.0, '旧距离': inf, '边': ('西站', '商场')},
 {'动作': '确认弹出', '节点': '东站', '距离': 7.0},
 {'动作': '松弛', '新距离': 9.0, '旧距离': 12.0, '边': ('东站', '商场')},
 {'动作': '松弛', '新距离': 10.0, '旧距离': inf, '边': ('东站', '学校')},
 {'动作': '跳过陈旧项', '堆中距离': 8.0, '最新距离': 7.0, '节点': '东站'},
 {'动作': '确认弹出'

## 逐样本结果与结果解读

Dijkstra 允许多走一条边来换取更短分钟，例如总仓先到南枢纽，再经西站到医院。结果表同时保留路径、分钟与相对最优的 regret；本例最优值由人工穷举小图核验。

In [4]:
dijkstra_rows = []  # 创建逐请求最短路结果表
for request in requests:  # 遍历六个相同配送请求
    path, minutes, ledger = dijkstra(request["起点"], request["终点"])  # 实际运行加权最短路
    baseline_minutes = next(row["真实分钟"] for row in baseline_rows if row["请求"] == request["id"])  # 读取同请求的 BFS 成本
    dijkstra_rows.append({"请求": request["id"], "BFS分钟": baseline_minutes, "Dijkstra路径": path, "Dijkstra分钟": minutes, "期望": request["最优分钟"], "BFS regret": baseline_minutes - request["最优分钟"], "Dijkstra正确": minutes == request["最优分钟"]})  # 保存同指标逐样本对照
baseline_exact = sum(row["真实分钟"] == row["最优分钟"] for row in baseline_rows)  # 统计 BFS 恰好最优的请求数
dijkstra_exact = sum(row["Dijkstra正确"] for row in dijkstra_rows)  # 统计 Dijkstra 命中人工最优值的请求数
print("逐请求路径与分钟对照：")  # 输出结果表标题
pprint(dijkstra_rows)  # 展示 BFS 和 Dijkstra 在每个请求上的差异
print(f"最优路线命中从 {baseline_exact}/{len(requests)} 提升到 {dijkstra_exact}/{len(requests)}")  # 输出同口径汇总结果

逐请求路径与分钟对照：
[{'BFS regret': 6,
  'BFS分钟': 14,
  'Dijkstra分钟': 8.0,
  'Dijkstra正确': True,
  'Dijkstra路径': ['总仓', '南枢纽', '西站', '医院'],
  '期望': 8,
  '请求': 'R1'},
 {'BFS regret': 2,
  'BFS分钟': 11,
  'Dijkstra分钟': 9.0,
  'Dijkstra正确': True,
  'Dijkstra路径': ['总仓', '南枢纽', '西站', '东站', '商场'],
  '期望': 9,
  '请求': 'R2'},
 {'BFS regret': 0,
  'BFS分钟': 7,
  'Dijkstra分钟': 7.0,
  'Dijkstra正确': True,
  'Dijkstra路径': ['北枢纽', '东站', '商场'],
  '期望': 7,
  '请求': 'R3'},
 {'BFS regret': 1,
  'BFS分钟': 9,
  'Dijkstra分钟': 8.0,
  'Dijkstra正确': True,
  'Dijkstra路径': ['南枢纽', '西站', '东站', '学校'],
  '期望': 8,
  '请求': 'R4'},
 {'BFS regret': 0,
  'BFS分钟': 4,
  'Dijkstra分钟': 4.0,
  'Dijkstra正确': True,
  'Dijkstra路径': ['西站', '东站', '学校'],
  '期望': 4,
  '请求': 'R5'},
 {'BFS regret': 12,
  'BFS分钟': 18,
  'Dijkstra分钟': 6.0,
  'Dijkstra正确': True,
  'Dijkstra路径': ['学校', '东站', '西站', '医院'],
  '期望': 6,
  '请求': 'R6'}]
最优路线命中从 2/6 提升到 6/6


## 失败案例：节点第一次入堆就标记 visited

总仓会先发现到北枢纽的 4 分钟路线，稍后才发现经南枢纽只需 3 分钟。错误实现若在第一次入堆就把北枢纽标记为 visited，就禁止这次改进，并继续传播错误距离。正确做法是在最小距离弹出时确认节点，同时允许松弛产生多个堆项。

In [5]:
def wrong_dijkstra_enqueue_visited(start, goal):  # 定义入堆即标记访问的常见错误实现
    heap = [(0.0, start, [start])]  # 初始化错误算法的距离、节点和路径
    visited = {start}  # 错误地把发现集合当成最终集合
    while heap:  # 持续弹出当前候选
        distance, current, path = heapq.heappop(heap)  # 取出堆中累计距离最小的候选
        if current == goal:  # 到达目标时返回受污染结果
            return path, distance  # 返回错误路径和成本
        for neighbor, weight in adjacency[current]:  # 遍历当前节点的道路
            if neighbor not in visited:  # 已入堆节点永远不允许被更短路径更新
                visited.add(neighbor)  # 在距离尚未最终确定时过早标记
                heapq.heappush(heap, (distance + weight, neighbor, path + [neighbor]))  # 把首次发现路线压入堆
    return [], float("inf")  # 不可达时返回空结果
wrong_path, wrong_minutes = wrong_dijkstra_enqueue_visited("总仓", "医院")  # 复现过早 visited 导致的次优路线
fixed_path, fixed_minutes, _ = dijkstra("总仓", "医院")  # 用允许重复松弛的正确实现求解
print("失败案例：入堆即 visited", {"路径": wrong_path, "分钟": wrong_minutes})  # 展示错误算法的路线和成本
print("修正：弹出最新最短项才确认", {"路径": fixed_path, "分钟": fixed_minutes})  # 展示正确算法的改进结果

失败案例：入堆即 visited {'路径': ['总仓', '南枢纽', '医院'], '分钟': 9.0}
修正：弹出最新最短项才确认 {'路径': ['总仓', '南枢纽', '西站', '医院'], '分钟': 8.0}


## 生产差距

真实路网包含单行、闭路、容量、时间依赖权重和持续更新，通常还需 A*、多目标路径或动态图算法。道路权重必须非负且版本一致；服务侧要限制搜索范围、设置超时、监控不可达率与 ETA 偏差，并把实际行程回流校准权重。

In [6]:
assert len(requests) == 6  # 验证案例包含六个真实语义配送请求
assert dijkstra_exact == len(requests)  # 验证手写最短路命中全部人工核验成本
assert dijkstra_exact > baseline_exact  # 验证加权算法优于最少跳数基线
assert example_minutes == 8  # 验证示例请求的最短分钟正确
assert wrong_minutes > fixed_minutes  # 验证过早 visited 确实产生次优成本
assert fixed_path == ["总仓", "南枢纽", "西站", "医院"]  # 验证正确实现重建预期最短路径
print("最小回归测试通过：加权最短路、路径重建与 visited 修复均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：加权最短路、路径重建与 visited 修复均满足预期
